# 04 — IRS Form 990 Financial Feature Engineering
### Final documented pipeline · full rebuild **or** repair existing extraction

**Purpose.** Convert Notebook 03's nested Form 990 XML trees into auditable filing-level financial, compensation, governance, and data-quality features; then match Notebook 02's canonical organization-year filings.

**Two supported execution modes**

- **`MODE = "repair"` (default):** Reuse the **50 existing financial partitions** produced by the earlier Notebook 04, scan the source `xml_tree_json` only for the corrected functional-expense section, and preserve all other extracted columns. **Use this mode now.**
- **`MODE = "full"`:** Rebuild all filing-level fields from Notebook 03's saved XML Parquet files using the corrected extraction logic. This is the **complete, reproducible from-scratch extraction** for project documentation.

**Critical correction.** Program, management, and fundraising expenses must come from the **same** `Return → ReturnData → IRS990 → TotalFunctionalExpensesGrp`. `TotalProgramServiceExpensesAmt` is **not** interchangeable with the `ProgramServicesAmt` in that group. The reported functional total and the summary total are kept separately, and differences are flagged—not silently fixed.

**Scope:** standard Form 990 (`IRS990`) only. Form 990-EZ, 990-PF, and 990-T were ingested in Notebook 03 but require separate form-specific feature mapping. **Financial outliers are not evidence of fraud.** Significant diversion is a self-reported disclosure, not a verified fraud label.

## 1 — Environment, paths, and run settings

In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SOURCE = ROOT / "data/staging/other_990_xml_partitions/full"
CANONICAL = ROOT / "data/cleaned/form990_organization_year_provisional_2023_2025.parquet"
OUT = ROOT / "data/analytics/financial_features"
LEGACY_PARTS = OUT / "filing_partitions_v2"  # 50 partitions from original expanded Notebook 04
REPAIR_PARTS = OUT / "functional_expense_corrections_v1"
FULL_PARTS = OUT / "filing_partitions_v4_full"
for folder in (OUT, REPAIR_PARTS, FULL_PARTS):
    folder.mkdir(parents=True, exist_ok=True)

FILES = sorted(SOURCE.glob("*/irs990.parquet"))
assert FILES, f"No standard Form 990 source partitions in {SOURCE}"
assert CANONICAL.exists(), f"Notebook 02 canonical file missing: {CANONICAL}"
ARCHIVE_NAMES = {p.parent.name for p in FILES}

# CHANGE TO "full" ONLY when you intend to rebuild from scratch.
MODE = "repair"              # "repair" or "full"
RUN_EXTRACTION = True        # Set False to only use previously completed partitions
OVERWRITE_PARTITIONS = False # Leave False to resume; True only after code/mapping changes
BATCH_SIZE = 250

assert MODE in {"repair", "full"}
print("Mode:", MODE)
print("Source archives:", len(FILES))
print("Existing original partitions:", len(list(LEGACY_PARTS.glob("*.parquet"))))
print("Existing repair partitions:", len(list(REPAIR_PARTS.glob("*.parquet"))))
print("Existing full-v4 partitions:", len(list(FULL_PARTS.glob("*.parquet"))))
print("Canonical:", CANONICAL)

## 2 — XML navigation, safe parsing, and source field mappings

In [ ]:
def local_name(tag):
    return str(tag).split("}")[-1].split(":")[-1]

def children_named(node, tag):
    if not isinstance(node, dict):
        return []
    return [x for x in node.get("children", []) if local_name(x.get("tag", "")) == tag]

def direct_child(node, tag):
    found = children_named(node, tag)
    return found[0] if len(found) == 1 else None

def main_990(xml_text):
    try:
        root = json.loads(xml_text)
    except (TypeError, ValueError):
        return None
    return direct_child(direct_child(root, "ReturnData"), "IRS990")

def parse_number(value):
    if value is None or isinstance(value, bool):
        return np.nan
    s = str(value).strip().replace(",", "").replace("$", "")
    if s.startswith("(") and s.endswith(")"):
        s = "-" + s[1:-1]
    if not s:
        return np.nan
    try:
        x = float(s)
        return x if math.isfinite(x) else np.nan
    except (TypeError, ValueError):
        return np.nan

def unique_direct_value(node, aliases, numeric=True):
    matches = [x.get("text") for x in (node or {}).get("children", [])
               if local_name(x.get("tag", "")) in aliases]
    if not matches:
        return (np.nan if numeric else None), "missing"
    values = [parse_number(x) if numeric else str(x).strip() for x in matches]
    values = [x for x in values if pd.notna(x) and x != ""]
    if not values:
        return (np.nan if numeric else None), "unparseable"
    if len(set(map(str, values))) > 1:
        return (np.nan if numeric else None), "ambiguous"
    return values[0], "ok"

def boolean_value(node, aliases):
    val, status = unique_direct_value(node, aliases, numeric=False)
    if status != "ok":
        return np.nan, status
    val = str(val).lower()
    if val in ("true", "1", "yes"):
        return True, "ok"
    if val in ("false", "0", "no"):
        return False, "ok"
    return np.nan, "invalid_boolean"

# All these values are DIRECT children of IRS990 (not supplementary schedules).
FINANCIAL = {
    "revenue_cy": ["CYTotalRevenueAmt"],
    "revenue_py": ["PYTotalRevenueAmt"],
    "expenses_cy": ["CYTotalExpensesAmt"],
    "expenses_py": ["PYTotalExpensesAmt"],
    "contributions_cy": ["CYContributionsGrantsAmt"],
    "contributions_py": ["PYContributionsGrantsAmt"],
    "program_revenue_cy": ["CYProgramServiceRevenueAmt"],
    "investment_income_cy": ["CYInvestmentIncomeAmt"],
    "grants_paid_cy": ["CYGrantsAndSimilarPaidAmt"],
    "assets_boy": ["TotalAssetsBOYAmt"],
    "assets_eoy": ["TotalAssetsEOYAmt"],
    "liabilities_boy": ["TotalLiabilitiesBOYAmt"],
    "liabilities_eoy": ["TotalLiabilitiesEOYAmt"],
    "net_assets_boy": ["NetAssetsOrFundBalancesBOYAmt"],
    "net_assets_eoy": ["NetAssetsOrFundBalancesEOYAmt"],
}
GOVERNANCE_COUNTS = {
    "voting_board_members": ["VotingMembersGoverningBodyCnt"],
    "independent_board_members": ["VotingMembersIndependentCnt"],
    "employees_count": ["TotalEmployeeCnt"],
    "volunteers_count": ["TotalVolunteersCnt"],
}
GOVERNANCE_FLAGS = {
    "conflict_of_interest_policy": ["ConflictOfInterestPolicyInd"],
    "whistleblower_policy": ["WhistleblowerPolicyInd"],
    "document_retention_policy": ["DocumentRetentionPolicyInd"],
    "compensation_review_ceo": ["CompensationProcessCEOInd"],
}
FUNCTIONAL_TAGS = {
    "TotalAmt": "functional_expenses_total",
    "ProgramServicesAmt": "program_expenses",
    "ManagementAndGeneralAmt": "management_expenses",
    "FundraisingAmt": "fundraising_expenses",
}

## 3 — Authoritative functional-expense extraction and compensation logic

In [ ]:
def extract_functional(form):
    out = {"functional_group_status": "missing_group"}
    out.update({col: np.nan for col in FUNCTIONAL_TAGS.values()})
    groups = children_named(form, "TotalFunctionalExpensesGrp")
    if len(groups) != 1:
        out["functional_group_status"] = "missing_group" if not groups else "multiple_groups"
        return out
    out["functional_group_status"] = "ok"
    group = groups[0]
    for tag, col in FUNCTIONAL_TAGS.items():
        matches = children_named(group, tag)
        if len(matches) == 1:
            out[col] = parse_number(matches[0].get("text"))
        elif len(matches) > 1:
            out["functional_group_status"] = "duplicate_amount_tag"
    return out

def extract_compensation(form):
    # Part VII, Section A person rows. This is not total organization payroll.
    groups = children_named(form, "Form990PartVIISectionAGrp")
    out = {
        "part_vii_person_rows": len(groups),
        "part_vii_compensation_rows": 0,
        "part_vii_compensation_incomplete_rows": 0,
        "part_vii_compensation_from_org": np.nan,
        "part_vii_compensation_from_related": np.nan,
        "part_vii_other_compensation": np.nan,
        "part_vii_compensation_total": np.nan,
    }
    if not groups:
        return out
    sums = [0.0, 0.0, 0.0]
    counted = 0
    incomplete = 0
    for group in groups:
        vals = [unique_direct_value(group, [tag])[0] for tag in (
            "ReportableCompFromOrgAmt", "ReportableCompFromRltdOrgAmt", "OtherCompensationAmt"
        )]
        if any(pd.isna(v) for v in vals):
            incomplete += 1
            continue
        sums = [a + b for a, b in zip(sums, vals)]
        counted += 1
    out["part_vii_compensation_rows"] = counted
    out["part_vii_compensation_incomplete_rows"] = incomplete
    if counted:
        for key, val in zip((
            "part_vii_compensation_from_org",
            "part_vii_compensation_from_related",
            "part_vii_other_compensation",
        ), sums):
            out[key] = val
        out["part_vii_compensation_total"] = sum(sums)
    return out

def extract_record(row):
    out = {key: row.get(key) for key in (
        "publication_year", "archive_name", "source_xml", "filing_id", "form_type",
        "ein", "tax_year", "tax_period_begin", "tax_period_end",
        "return_type", "organization_name"
    )}
    form = main_990(row.get("xml_tree_json"))
    out["main_990_found"] = form is not None
    if form is None:
        return out
    for field, tags in FINANCIAL.items():
        out[field], out[field + "_status"] = unique_direct_value(form, tags)
    for field, tags in GOVERNANCE_COUNTS.items():
        out[field], out[field + "_status"] = unique_direct_value(form, tags)
    for field, tags in GOVERNANCE_FLAGS.items():
        out[field], out[field + "_status"] = boolean_value(form, tags)
    out.update(extract_functional(form))
    out.update(extract_compensation(form))
    return out

def extract_functional_patch(row):
    # Same authoritative parser, used to repair the previously extracted v2 partitions.
    form = main_990(row.get("xml_tree_json"))
    result = {"filing_id": str(row["filing_id"])}
    result.update(extract_functional(form))
    return result

## 4 — Sample tests: verify original values and corrected functional expenses

In [ ]:
sample = next(pq.ParquetFile(FILES[0]).iter_batches(batch_size=10)).to_pandas()
sample_features = pd.DataFrame(extract_record(row) for row in sample.to_dict("records"))
display(sample_features[[
    "filing_id", "revenue_cy", "expenses_cy", "assets_eoy",
    "program_expenses", "management_expenses", "fundraising_expenses",
    "part_vii_compensation_total", "voting_board_members"
]].head(10))
first = sample_features.iloc[0]
assert first["revenue_cy"] == 392303
assert first["expenses_cy"] == 212049
assert first["program_expenses"] == 169989
assert first["management_expenses"] == 13733
assert first["fundraising_expenses"] == 28327
assert first["part_vii_compensation_total"] == 65480
print("Known first-filing tests passed.")

## 5 — Run the chosen extraction mode (resumable)

**Your current run:** `MODE="repair"`. This **does not rerun the original complete extractor**. It scans only `filing_id` and `xml_tree_json` from Notebook 03 to correct four functional-expense amounts in the 50 previously saved financial partitions.

**Documentation / future clean rebuild:** `MODE="full"` runs the complete corrected `extract_record()` for each source filing and writes a separate `filing_partitions_v4_full` dataset. It does **not** overwrite the old v2 partitions.

Each archive is written atomically, so an interrupted run can resume with `OVERWRITE_PARTITIONS=False`. Changing extraction logic requires rebuilding affected saved partitions; do not reuse stale partitions after a mapping change.

In [ ]:
if MODE == "repair":
    originals = sorted(LEGACY_PARTS.glob("*.parquet"))
    assert {p.stem for p in originals} == ARCHIVE_NAMES, (
        "Repair mode needs the 50 original financial partitions in " + str(LEGACY_PARTS)
    )

if RUN_EXTRACTION:
    destination_folder = REPAIR_PARTS if MODE == "repair" else FULL_PARTS
    for i, src in enumerate(FILES, 1):
        archive = src.parent.name
        dest = destination_folder / f"{archive}.parquet"
        if dest.exists() and not OVERWRITE_PARTITIONS:
            print(f"[{i}/{len(FILES)}] SKIP {archive}", flush=True)
            continue

        rows = []
        pf = pq.ParquetFile(src)
        selected_columns = (["filing_id", "xml_tree_json"] if MODE == "repair" else None)
        for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=selected_columns):
            for row in batch.to_pandas().to_dict("records"):
                rows.append(
                    extract_functional_patch(row) if MODE == "repair"
                    else extract_record(row)
                )
        part = pd.DataFrame.from_records(rows)
        assert len(part) > 0, f"No rows in {archive}"
        part["filing_id"] = part["filing_id"].astype(str)
        assert part["filing_id"].is_unique, f"Duplicate filing IDs within {archive}"

        if MODE == "repair":
            legacy = pq.read_table(
                LEGACY_PARTS / f"{archive}.parquet", columns=["filing_id"]
            ).column("filing_id").to_pandas().astype(str)
            assert set(part["filing_id"]) == set(legacy), f"Repair ID mismatch: {archive}"

        temp = dest.with_suffix(".tmp.parquet")
        part.to_parquet(temp, index=False)
        temp.replace(dest)
        print(f"[{i}/{len(FILES)}] SAVED {archive}: {len(part):,} rows", flush=True)
else:
    print("Extraction disabled; using saved partitions.")

## 6 — Load the extracted dataset and apply corrections (if repair mode)

In [ ]:
def validate_partition_set(folder):
    found = sorted(folder.glob("*.parquet"))
    assert {p.stem for p in found} == ARCHIVE_NAMES, (
        f"Expected {len(ARCHIVE_NAMES)} archive partitions in {folder}; found {len(found)}"
    )
    return found

if MODE == "repair":
    original_files = validate_partition_set(LEGACY_PARTS)
    patch_files = validate_partition_set(REPAIR_PARTS)
    original = pd.concat((pd.read_parquet(p) for p in original_files), ignore_index=True)
    patches = pd.concat((pd.read_parquet(p) for p in patch_files), ignore_index=True)
    original["filing_id"] = original["filing_id"].astype(str)
    patches["filing_id"] = patches["filing_id"].astype(str)
    assert original["filing_id"].is_unique and patches["filing_id"].is_unique
    assert len(original) == len(patches), "Patch and original row counts differ"

    # Remove ONLY the incorrect functional-expense columns and their old statuses.
    replace_cols = list(FUNCTIONAL_TAGS.values())
    stale_cols = replace_cols + [c + "_status" for c in replace_cols] + ["functional_group_status"]
    f = original.drop(columns=stale_cols, errors="ignore").merge(
        patches, on="filing_id", how="left", validate="one_to_one", indicator=True
    )
    assert f["_merge"].eq("both").all(), "Unmatched correction records"
    f = f.drop(columns="_merge")
    for col in replace_cols:
        f[col + "_status"] = np.where(f[col].notna(), "ok", "missing")
else:
    full_files = validate_partition_set(FULL_PARTS)
    f = pd.concat((pd.read_parquet(p) for p in full_files), ignore_index=True)
    f["filing_id"] = f["filing_id"].astype(str)

assert f["filing_id"].is_unique, "Duplicate filing IDs across partitions"
assert f["filing_id"].notna().all()
print("Loaded filings:", f"{len(f):,}")
print("Functional expense group status:")
print(f["functional_group_status"].value_counts(dropna=False).to_string())
coverage_fields = ["revenue_cy", "expenses_cy", "program_expenses",
                   "management_expenses", "fundraising_expenses", "functional_expenses_total"]
coverage = pd.DataFrame({
    "field": coverage_fields,
    "nonmissing": [int(f[x].notna().sum()) for x in coverage_fields],
    "coverage_pct": [round(100 * f[x].notna().mean(), 2) for x in coverage_fields],
})
display(coverage)

## 7 — Validate known filings and functional-expense reconciliation

In [ ]:
known = {
    "202300109349300000": {
        "expenses_cy": 212049,
        "program_expenses": 169989,
        "management_expenses": 13733,
        "fundraising_expenses": 28327,
    },
    "202300149349300705": {
        "expenses_cy": 570775,
        "program_expenses": 510990,
        "management_expenses": 37657,
        "fundraising_expenses": 22400,
    },
}
for filing_id, expected in known.items():
    matched = f.loc[f["filing_id"] == filing_id]
    assert len(matched) == 1, f"Expected exactly one row for {filing_id}"
    for field, value in expected.items():
        assert matched.iloc[0][field] == value, (
            f"{filing_id}: {field} expected {value}, got {matched.iloc[0][field]}"
        )
print("Known-filing tests passed.")

components = ["program_expenses", "management_expenses", "fundraising_expenses"]
f["functional_component_sum"] = f[components].sum(axis=1, min_count=3)
f["functional_component_difference"] = (
    f["functional_expenses_total"] - f["functional_component_sum"]
)
f["summary_vs_functional_difference"] = (
    f["expenses_cy"] - f["functional_expenses_total"]
)
f["functional_component_mismatch_flag"] = (
    f["functional_component_difference"].notna()
    & f["functional_component_difference"].abs().gt(1)
)
f["summary_vs_functional_mismatch_flag"] = (
    f["summary_vs_functional_difference"].notna()
    & f["summary_vs_functional_difference"].abs().gt(1)
)
print("Component mismatches (>$1):", int(f["functional_component_mismatch_flag"].sum()))
print("Summary vs functional mismatches (>$1):", int(f["summary_vs_functional_mismatch_flag"].sum()))
display(f.loc[f["filing_id"].isin(known), [
    "filing_id", "expenses_cy", "functional_expenses_total",
    *components, "functional_component_difference", "summary_vs_functional_difference"
]])

In [ ]:
mismatch = f.loc[
    f["functional_component_mismatch_flag"],
    [
        "filing_id",
        "expenses_cy",
        "functional_expenses_total",
        "program_expenses",
        "management_expenses",
        "fundraising_expenses",
        "functional_component_sum",
        "functional_component_difference",
    ]
].copy()

print("Total mismatches:", len(mismatch))

print("\nDifference statistics:")
print(mismatch["functional_component_difference"].describe())

print("\nMost common differences:")
print(
    mismatch["functional_component_difference"]
    .value_counts()
    .head(15)
)

print("\nLargest absolute differences:")
display(
    mismatch.reindex(
        mismatch["functional_component_difference"]
        .abs()
        .sort_values(ascending=False)
        .index
    ).head(10)
)

In [ ]:
components = [
    "program_expenses",
    "management_expenses",
    "fundraising_expenses"
]

all_zero = f[components].eq(0).all(axis=1)

zero_component_issue = (
    all_zero
    & f["functional_expenses_total"].gt(0)
)

print("All three components zero, total > 0:",
      zero_component_issue.sum())

print("Of the 80,526 mismatches, how many are all-zero:")
print(
    (
        zero_component_issue
        & f["functional_component_mismatch_flag"]
    ).sum()
)

print("Remaining mismatches:")
print(
    (
        ~zero_component_issue
        & f["functional_component_mismatch_flag"]
    ).sum()
)

print("\nExamples:")
display(
    f.loc[zero_component_issue, [
        "filing_id",
        "functional_expenses_total",
        *components
    ]].head(10)
)

In [ ]:
# Identify incomplete or inconsistent functional expense allocations

components = [
    "program_expenses",
    "management_expenses",
    "fundraising_expenses"
]

# All three components are zero, despite positive reported total
f["functional_all_zero_with_positive_total_flag"] = (
    f[components].eq(0).all(axis=1)
    & f["functional_expenses_total"].gt(0)
)

# Components missing or not reconciling to reported total
f["functional_allocation_unreliable_flag"] = (
    f[components].isna().any(axis=1)
    | f["functional_expenses_total"].isna()
    | f["functional_component_mismatch_flag"]
)

# Only use functional allocation ratios when values reconcile
f["functional_allocation_valid_flag"] = (
    ~f["functional_allocation_unreliable_flag"]
    & f["functional_expenses_total"].gt(0)
)

print("All-zero with positive total:",
      f["functional_all_zero_with_positive_total_flag"].sum())

print("Unreliable allocations:",
      f["functional_allocation_unreliable_flag"].sum())

print("Valid allocations:",
      f["functional_allocation_valid_flag"].sum())

print("Valid allocation percentage:",
      round(100 * f["functional_allocation_valid_flag"].mean(), 2))

## 8 — Financial growth, operating performance, funding, expense, and governance features

Growth is a percentage of the **reported prior-year amount** (or beginning-of-year balance). A zero or missing denominator produces `NaN`, never infinity. Functional expense ratios use the IRS-reported `functional_expenses_total` as their denominator, not a different section's total.

**Interpretation:** `part_vii_compensation_total` is reported Part VII Section A compensation for rows with complete components—not the organization's total payroll. Compensation completeness is tracked separately.

In [ ]:
def ratio(numerator, denominator):
    a = pd.to_numeric(numerator, errors="coerce")
    b = pd.to_numeric(denominator, errors="coerce")
    return a.div(b.where(b.ne(0)))

for name, current, previous in [
    ("revenue", "revenue_cy", "revenue_py"),
    ("expenses", "expenses_cy", "expenses_py"),
    ("assets", "assets_eoy", "assets_boy"),
    ("liabilities", "liabilities_eoy", "liabilities_boy"),
    ("net_assets", "net_assets_eoy", "net_assets_boy"),
]:
    f[name + "_change_amount"] = f[current] - f[previous]
    f[name + "_change_pct"] = 100 * ratio(f[name + "_change_amount"], f[previous])
    f[name + "_prior_zero_flag"] = f[previous].notna() & f[previous].eq(0)

f["operating_surplus"] = f["revenue_cy"] - f["expenses_cy"]
f["operating_margin"] = ratio(f["operating_surplus"], f["revenue_cy"])
f["expense_to_revenue"] = ratio(f["expenses_cy"], f["revenue_cy"])
f["contributions_share"] = ratio(f["contributions_cy"], f["revenue_cy"])
f["program_revenue_share"] = ratio(f["program_revenue_cy"], f["revenue_cy"])
f["investment_income_share"] = ratio(f["investment_income_cy"], f["revenue_cy"])
# Calculate functional expense ratios only for valid allocations

valid = f["functional_allocation_valid_flag"]

f["program_expense_share"] = (
    ratio(f["program_expenses"], f["functional_expenses_total"])
    .where(valid)
)

f["management_expense_share"] = (
    ratio(f["management_expenses"], f["functional_expenses_total"])
    .where(valid)
)

f["fundraising_expense_share"] = (
    ratio(f["fundraising_expenses"], f["functional_expenses_total"])
    .where(valid)
)
f["liabilities_to_assets"] = ratio(f["liabilities_eoy"], f["assets_eoy"])
f["independent_board_share"] = ratio(f["independent_board_members"], f["voting_board_members"])
f["compensation_to_expenses"] = ratio(f["part_vii_compensation_total"], f["expenses_cy"])
display(f[[
    "revenue_change_pct", "expenses_change_pct", "assets_change_pct",
    "operating_margin", "program_expense_share",
    "management_expense_share", "fundraising_expense_share"
]].describe())

## 9 — Data-quality flags and audit summary

In [ ]:
core = ["revenue_cy", "expenses_cy", "assets_eoy", "liabilities_eoy"]
f["missing_core_financial_count"] = f[core].isna().sum(axis=1)
f["functional_expense_missing_flag"] = (
    f["functional_expenses_total"].isna() | f["functional_component_sum"].isna()
)
f["negative_assets_flag"] = f["assets_eoy"].notna() & f["assets_eoy"].lt(0)
f["negative_net_assets_flag"] = f["net_assets_eoy"].notna() & f["net_assets_eoy"].lt(0)
f["negative_revenue_flag"] = f["revenue_cy"].notna() & f["revenue_cy"].lt(0)
f["balance_sheet_residual"] = f["assets_eoy"] - f["liabilities_eoy"] - f["net_assets_eoy"]
f["balance_sheet_mismatch_flag"] = (
    f["balance_sheet_residual"].notna() & f["balance_sheet_residual"].abs().gt(1)
)
f["extreme_revenue_growth_review_flag"] = (
    f["revenue_change_pct"].notna() & f["revenue_change_pct"].abs().gt(500)
)
f["compensation_partial_flag"] = f["part_vii_compensation_incomplete_rows"].fillna(0).gt(0)
f["board_count_inconsistent_flag"] = (
    f["independent_board_members"].notna()
    & f["voting_board_members"].notna()
    & (f["independent_board_members"] > f["voting_board_members"])
)
flags = [x for x in f.columns if x.endswith("_flag")]
flag_summary = pd.DataFrame({
    "flag": flags,
    "count": [int(f[x].fillna(False).sum()) for x in flags]
}).sort_values("count", ascending=False)
display(flag_summary)
print("These are data-quality/review flags, NOT verified fraud indicators.")

## 10 — Strict canonical organization-year match (Notebook 02)

The broad filing extraction is **not** the final modeling population. Match to Notebook 02's provisional canonical selection using a unique filing-level identifier, preserving the canonical dataset's organization-year decisions and disclosure information. Stop if the identifier or coverage is inconsistent; never silently join on EIN alone.

In [ ]:
canonical = pd.read_parquet(CANONICAL)
print("Canonical rows:", len(canonical))
identifier_candidates = ["filing_id", "source_xml", "source_file", "xml_filename"]
shared = [key for key in identifier_candidates if key in canonical.columns and key in f.columns]
join_key = next((
    key for key in shared
    if canonical[key].notna().all()
    and canonical[key].is_unique
    and f[key].notna().all()
    and f[key].is_unique
), None)
if join_key is None:
    raise ValueError(
        f"No unique, nonmissing shared filing ID. Shared columns: {shared}. "
        "Inspect Notebook 02's canonical schema before saving."
    )
print("Join key:", join_key)
feature_only = f.drop(columns=[
    key for key in f.columns if key in canonical.columns and key != join_key
])
canonical_features = canonical.merge(
    feature_only, on=join_key, how="left", validate="one_to_one", indicator=True
)
print("Join results:")
print(canonical_features["_merge"].value_counts(dropna=False).to_string())
assert canonical_features["_merge"].eq("both").all(), (
    "Some canonical filings are unmatched. Investigate before saving."
)
canonical_features = canonical_features.drop(columns="_merge")
assert len(canonical_features) == len(canonical)
print("Canonical feature rows:", f"{len(canonical_features):,}")

## 11 — Save versioned outputs and audit report

Writes **new v4 files** without modifying the original v2 financial partitions. Both modes produce the same analysis-ready output names, but the audit records the mode used. Run this section only after validation and canonical matching succeed.

In [ ]:
assert len(f) > 0
assert f["filing_id"].is_unique
assert len(canonical_features) == len(canonical)

filing_output = OUT / "form990_filing_financial_features_v4.parquet"
canonical_output = OUT / "form990_canonical_organization_year_financial_features_v4.parquet"
audit_output = OUT / "04_financial_feature_engineering_v4_audit.json"

f.to_parquet(filing_output, index=False)
canonical_features.to_parquet(canonical_output, index=False)

audit = {
    "pipeline": "04_financial_feature_engineering_v4",
    "mode": MODE,
    "source_archives": len(FILES),
    "filing_rows": int(len(f)),
    "canonical_rows": int(len(canonical_features)),
    "canonical_join_key": join_key,
    "original_partitions_reused": len(list(LEGACY_PARTS.glob("*.parquet"))) if MODE == "repair" else 0,
    "functional_expense_patch_partitions": len(list(REPAIR_PARTS.glob("*.parquet"))) if MODE == "repair" else 0,
    "functional_group_status": {str(k): int(v) for k, v in f["functional_group_status"].value_counts(dropna=False).items()},
    "functional_component_mismatches_gt_1": int(f["functional_component_mismatch_flag"].sum()),
    "summary_vs_functional_mismatches_gt_1": int(f["summary_vs_functional_mismatch_flag"].sum()),
    "coverage_pct": {key: round(100 * f[key].notna().mean(), 2)
                     for key in ["revenue_cy", "expenses_cy", "program_expenses",
                                 "management_expenses", "fundraising_expenses"]},
    "limitations": [
        "Standard IRS990 only; 990-EZ, 990-PF and 990-T need separate standardization",
        "Prior-year amounts embedded in a return may be revised; check availability for prediction-time modeling",
        "Self-reported significant diversion is not verified fraud",
        "Financial and data-quality flags are not proof of misconduct",
        "Part VII compensation covers listed person rows, not total organizational payroll",
        "Repair mode retains original v2 nonfunctional extraction logic and corrects functional expense fields only",
    ],
}
audit_output.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
print("Saved filing features:", filing_output)
print("Saved canonical features:", canonical_output)
print("Saved audit report:", audit_output)
print("Notebook 04 complete.")

## 12 — Data lineage, interpretation, and next steps

**Lineage:** 50 IRS XML archives → Notebook 03 (`irs990.parquet`, including `xml_tree_json`) → Notebook 04 extraction or targeted repair → filing-level financial features → Notebook 02 canonical organization-year join → Notebook 05 anomaly detection.

**Why the correction matters:** `TotalProgramServiceExpensesAmt` can represent a different section of Form 990 than the functional expense allocation. The correct four values are children of `TotalFunctionalExpensesGrp`:

- `TotalAmt` → `functional_expenses_total`
- `ProgramServicesAmt` → `program_expenses`
- `ManagementAndGeneralAmt` → `management_expenses`
- `FundraisingAmt` → `fundraising_expenses`

**Reconciliation:** The sum of the three functional expense components may disagree with the reported functional total, and that total may disagree with the summary `CYTotalExpensesAmt`. We preserve all values and explicitly flag both differences. Do **not** derive management expenses as a residual or replace IRS-reported amounts.

**Remaining work:** Inspect final coverage and reconciliation counts; build Notebook 04B for other form types if desired; use Notebook 05 for peer-group anomaly detection. For any future supervised significant-diversion prediction, apply appropriate temporal feature availability and label-leakage controls.

### Data Quality, Extraction Corrections, and Limitations

This notebook processed **1,314,954 IRS Form 990 filings** and generated financial, compensation, governance, and data-quality features for downstream anomaly detection.

**Extraction correction and repair mode:** During validation, the original extraction was found to use incorrect XML fields for certain functional expense categories. The pipeline was corrected to extract program, management, and fundraising expenses directly from `TotalFunctionalExpensesGrp`. Rather than repeating the entire extraction across 50 archives, the notebook was executed in **repair mode**, which preserved previously extracted financial and governance data while reprocessing only the affected functional expense fields. The notebook also supports a full extraction mode for future reproducibility.

**Data-quality findings:**
- **80,526 filings (6.1%)** had functional expense components that differed from their reported functional expense total by more than $1.
- **63,317 filings** reported zero expenses across all three functional categories despite having positive total functional expenses, suggesting potentially incomplete or unreliable expense allocations.
- **534 filings** had discrepancies between their summary total expenses and functional expense totals.

To address these issues, the pipeline preserves the original reported values, creates explicit data-quality flags, and excludes unreliable functional expense allocations from ratio-based features. No missing or inconsistent financial amounts are automatically imputed or adjusted.

**Implications for downstream analysis:** Revenue, total expenses, assets, liabilities, and other independently reported financial fields remain available for anomaly detection. Functional expense ratios are used only when the reported allocations are complete and reconcile within the defined tolerance. Data-quality inconsistencies may warrant further investigation but should **not be interpreted as evidence of fraud**. Similarly, significant-diversion disclosures represent self-reported information rather than independently verified fraud.

These safeguards improve the transparency, reproducibility, and reliability of the financial features used in subsequent notebooks.